# Inverse RL: Reading Goals from Behavior

Companion notebook for Tutorial 3, Chapter 23. Runs every code block from the chapter.

Chapter: https://josephausterweil.github.io/probintro/intro2/23_inverse_rl_goal_inference/

In [ ]:
# Colab setup
!pip -q install "genjax==0.10.3" "jax==0.5.3" 2>/dev/null

In [ ]:
import jax.numpy as jnp
from jax import random
from genjax import gen, categorical, ChoiceMap

NROWS, NCOLS, NA = 3, 3, 4
NS = NROWS * NCOLS
START = 0                                   # (0,0) bottom-left
GOALS = {"left": 6, "mid": 7, "right": 8}   # the three top-row cells
NAMES = list(GOALS)
GOAL_STATES = jnp.array([GOALS[g] for g in NAMES])
GAMMA, BETA = 0.9, 3.0                       # discount; rationality beta (chosen, not inferred)
UP, DOWN, LEFT, RIGHT = 0, 1, 2, 3

def step(s, a):                              # deterministic move, clamped at the walls
    r, c = s // NCOLS, s % NCOLS
    r = jnp.clip(jnp.where(a == UP, r + 1, jnp.where(a == DOWN, r - 1, r)), 0, NROWS - 1)
    c = jnp.clip(jnp.where(a == LEFT, c - 1, jnp.where(a == RIGHT, c + 1, c)), 0, NCOLS - 1)
    return r * NCOLS + c

TRANS = jnp.array([[int(step(s, a)) for a in range(NA)] for s in range(NS)])  # T[s,a] -> s'

def q_values_for_goal(goal):                 # value iteration for ONE goal (the forward planner)
    reward = (TRANS == goal).astype(jnp.float32)     # +1 for entering the goal cell
    is_goal = (jnp.arange(NS) == goal)
    V = jnp.zeros(NS)
    for _ in range(50):
        Q = reward + GAMMA * V[TRANS]
        V = jnp.where(is_goal, 0.0, jnp.max(Q, axis=1))
    return reward + GAMMA * V[TRANS]                  # Q_g(s,a), shape (NS, NA)

In [ ]:
LOGITS = BETA * jnp.stack([q_values_for_goal(g) for g in GOAL_STATES])   # (G, NS, NA)
PRIOR = jnp.ones(len(NAMES)) / len(NAMES)                                 # uniform over goals

@gen
def observer(states):                        # the generative model of a goal-seeker
    g = categorical(jnp.log(PRIOR)) @ "goal"             # 1. draw a hidden goal
    for t in range(len(states)):
        categorical(LOGITS[g, states[t]]) @ f"a_{t}"     # 2. act by goal g's softmax policy

def logsumexp(x):
    m = jnp.max(x); return m + jnp.log(jnp.sum(jnp.exp(x - m)))

def goal_posterior(states, actions):         # exact P(goal | actions) by enumerating goals
    states = jnp.asarray(states)
    logp = []
    for g in range(len(NAMES)):
        cm = ChoiceMap.d({"goal": g, **{f"a_{t}": int(actions[t]) for t in range(len(actions))}})
        score, _ = observer.assess(cm, (states,))        # log P(goal, actions)
        logp.append(score)
    return jnp.exp(jnp.array(logp) - logsumexp(jnp.array(logp)))   # normalize -> posterior

def states_visited(start, actions):          # the state at which each action is taken
    s, out = start, []
    for a in actions:
        out.append(int(s)); s = int(step(jnp.array(s), jnp.array(a)))
    return out

In [ ]:
actions = [RIGHT, RIGHT, UP, UP]             # (0,0)->(0,1)->(0,2)->(1,2)->(2,2): the right goal
states = states_visited(START, actions)
post = goal_posterior(states, actions)
print("observed actions:", ["UP DOWN LEFT RIGHT".split()[a] for a in actions])
for name, p in zip(NAMES, post):
    print(f"  P({name:5s} | actions) = {float(p):.3f}")

In [ ]:
print("step  " + "  ".join(f"{n:>5s}" for n in NAMES))
for k in range(1, len(actions) + 1):
    pk = goal_posterior(states[:k], actions[:k])
    print(f" {k:>2d}   " + "  ".join(f"{float(x):.3f}" for x in pk))

In [ ]:
for b in (0.1, 3.0, 8.0):
    LOGITS = b * jnp.stack([q_values_for_goal(g) for g in GOAL_STATES])
    pk = goal_posterior(states, actions)
    tag = {0.1: "near-random", 3.0: "noisy-rational", 8.0: "near-greedy"}[b]
    print(f"beta={b:>4} ({tag:>14}):  " + "  ".join(f"P({n})={float(x):.3f}" for n, x in zip(NAMES, pk)))